In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os
data_dir = '/content/drive/MyDrive/MIA_Project_2/Train'  # adjust path if yours is different
subjects = sorted(os.listdir(data_dir))
print(f"Found {len(subjects)} subjects")
print(subjects[:5])

Found 200 subjects
['subj001', 'subj002', 'subj003', 'subj004', 'subj005']


In [3]:
!pip install nibabel -q

In [4]:
import nibabel as nib
import numpy as np

test_subj = 'subj001'
img = nib.load(f'{data_dir}/{test_subj}/{test_subj}_image.nii.gz')
cath = nib.load(f'{data_dir}/{test_subj}/{test_subj}_catheter.nii.gz')

print(f"Image shape: {img.shape}")
print(f"Voxel spacing: {img.header.get_zooms()}")
print(f"Catheter mask unique values: {np.unique(cath.get_fdata())}")
print(f"Catheter voxel count: {int(cath.get_fdata().sum())}")

Image shape: (192, 224, 192)
Voxel spacing: (np.float32(1.0), np.float32(1.0), np.float32(1.0))
Catheter mask unique values: [0. 1. 2.]
Catheter voxel count: 1582


In [5]:
import json

for test_subj in ['subj001', 'subj002', 'subj003']:
    cath = nib.load(f'{data_dir}/{test_subj}/{test_subj}_catheter.nii.gz').get_fdata()
    artifact = nib.load(f'{data_dir}/{test_subj}/{test_subj}_artifact.nii.gz').get_fdata()

    with open(f'{data_dir}/{test_subj}/{test_subj}.json') as f:
        meta = json.load(f)

    print(f"\n{test_subj}:")
    print(f"  Catheter values: {np.unique(cath)}")
    print(f"  Value 1 count: {int((cath==1).sum())}")
    print(f"  Value 2 count: {int((cath==2).sum())}")
    print(f"  Artifact voxels: {int(artifact.sum())}")
    print(f"  Metadata: {meta}")


subj001:
  Catheter values: [0. 1. 2.]
  Value 1 count: 864
  Value 2 count: 359
  Artifact voxels: 11995
  Metadata: {'subject_id': 'subj001', 'hemisphere': 'right', 'catheter_length_mm': 55.28, 'spacing_zyx_mm': [1.0, 1.0, 1.0]}

subj002:
  Catheter values: [0. 1. 2.]
  Value 1 count: 573
  Value 2 count: 366
  Artifact voxels: 14073
  Metadata: {'subject_id': 'subj002', 'hemisphere': 'right', 'catheter_length_mm': 41.27, 'spacing_zyx_mm': [1.0, 1.0, 1.0]}

subj003:
  Catheter values: [0. 1. 2.]
  Value 1 count: 933
  Value 2 count: 391
  Artifact voxels: 16692
  Metadata: {'subject_id': 'subj003', 'hemisphere': 'right', 'catheter_length_mm': 60.93, 'spacing_zyx_mm': [1.0, 1.0, 1.0]}


In [6]:
subjects_with_catheter = []
subjects_without_catheter = []

for subj in subjects:
    cath = nib.load(f'{data_dir}/{subj}/{subj}_catheter.nii.gz').get_fdata()
    if cath.sum() > 0:
        subjects_with_catheter.append(subj)
    else:
        subjects_without_catheter.append(subj)

print(f"Subjects WITH catheter: {len(subjects_with_catheter)}")
print(f"Subjects WITHOUT catheter: {len(subjects_without_catheter)}")

Subjects WITH catheter: 200
Subjects WITHOUT catheter: 0


Set up training pipeline

In [2]:
import os
import json
import numpy as np
import nibabel as nib
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Check GPU memory if available
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Using device: cuda
GPU: Tesla T4
Memory: 15.6 GB


In [3]:
np.random.seed(42)
all_subjects = sorted(os.listdir(data_dir))

# 80/20 split, stratified by hemisphere
with open(f'{data_dir}/subj001/subj001.json') as f:
    pass  # just testing

left_subjs = []
right_subjs = []

for subj in all_subjects:
    with open(f'{data_dir}/{subj}/{subj}.json') as f:
        meta = json.load(f)
    if meta['hemisphere'] == 'left':
        left_subjs.append(subj)
    else:
        right_subjs.append(subj)

print(f"Left hemisphere: {len(left_subjs)}")
print(f"Right hemisphere: {len(right_subjs)}")

# Stratified split
np.random.shuffle(left_subjs)
np.random.shuffle(right_subjs)

n_left_train = int(0.8 * len(left_subjs))
n_right_train = int(0.8 * len(right_subjs))

train_subjects = left_subjs[:n_left_train] + right_subjs[:n_right_train]
val_subjects = left_subjs[n_left_train:] + right_subjs[n_right_train:]

print(f"\nTrain subjects: {len(train_subjects)}")
print(f"Val subjects: {len(val_subjects)}")

Left hemisphere: 10
Right hemisphere: 190

Train subjects: 160
Val subjects: 40


In [4]:
class CatheterPatchDataset(Dataset):
    def __init__(self, cache_dir, subject_ids, patch_size=64, patches_per_subject=32, augment=False):
        self.cache_dir = cache_dir
        self.subject_ids = subject_ids
        self.patch_size = patch_size
        self.patches_per_subject = patches_per_subject
        self.augment = augment
        self.p = patch_size // 2

        print("Scanning catheter coordinates...")
        self.coords = {}
        self.shapes = {}
        for subj in tqdm(subject_ids):
            mask = np.load(f'{cache_dir}/{subj}_catheter.npy')
            self.coords[subj] = np.array(np.where(mask > 0)).T
            self.shapes[subj] = mask.shape

    def __len__(self):
        return len(self.subject_ids) * self.patches_per_subject

    def __getitem__(self, idx):
        subj = self.subject_ids[idx // self.patches_per_subject]
        coords = self.coords[subj]
        D, H, W = self.shapes[subj]
        p = self.p

        # Fast numpy load instead of nii.gz
        img = np.load(f'{self.cache_dir}/{subj}_image.npy')
        mask = np.load(f'{self.cache_dir}/{subj}_catheter.npy')

        if np.random.rand() < 0.5 and len(coords) > 0:
            center = coords[np.random.randint(len(coords))]
        else:
            center = np.array([
                np.random.randint(p, D - p),
                np.random.randint(p, H - p),
                np.random.randint(p, W - p)
            ])

        center = np.clip(center, p, np.array([D, H, W]) - p - 1)
        z, y, x = center

        img_patch = img[z-p:z+p, y-p:y+p, x-p:x+p].copy()
        mask_patch = mask[z-p:z+p, y-p:y+p, x-p:x+p].copy()

        if self.augment:
            for axis in range(3):
                if np.random.rand() < 0.5:
                    img_patch = np.flip(img_patch, axis=axis).copy()
                    mask_patch = np.flip(mask_patch, axis=axis).copy()

        return (torch.tensor(img_patch).unsqueeze(0),
                torch.tensor(mask_patch).unsqueeze(0))

In [5]:
class ConvBlock3D(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1),
            nn.InstanceNorm3d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_ch, out_ch, 3, padding=1),
            nn.InstanceNorm3d(out_ch),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)


class UNet3D(nn.Module):
    def __init__(self, base_channels=16):
        super().__init__()
        b = base_channels

        # Encoder
        self.enc1 = ConvBlock3D(1, b)
        self.enc2 = ConvBlock3D(b, b*2)
        self.enc3 = ConvBlock3D(b*2, b*4)
        self.enc4 = ConvBlock3D(b*4, b*8)

        self.pool = nn.MaxPool3d(2)

        # Bottleneck
        self.bottleneck = ConvBlock3D(b*8, b*16)

        # Decoder
        self.up4 = nn.ConvTranspose3d(b*16, b*8, 2, stride=2)
        self.dec4 = ConvBlock3D(b*16, b*8)

        self.up3 = nn.ConvTranspose3d(b*8, b*4, 2, stride=2)
        self.dec3 = ConvBlock3D(b*8, b*4)

        self.up2 = nn.ConvTranspose3d(b*4, b*2, 2, stride=2)
        self.dec2 = ConvBlock3D(b*4, b*2)

        self.up1 = nn.ConvTranspose3d(b*2, b, 2, stride=2)
        self.dec1 = ConvBlock3D(b*2, b)

        self.tail = nn.Conv3d(b, 1, 1)

    def forward(self, x):
        # Encoder
        s1 = self.enc1(x)
        s2 = self.enc2(self.pool(s1))
        s3 = self.enc3(self.pool(s2))
        s4 = self.enc4(self.pool(s3))

        # Bottleneck
        x = self.bottleneck(self.pool(s4))

        # Decoder
        x = self.dec4(torch.cat([self.up4(x), s4], dim=1))
        x = self.dec3(torch.cat([self.up3(x), s3], dim=1))
        x = self.dec2(torch.cat([self.up2(x), s2], dim=1))
        x = self.dec1(torch.cat([self.up1(x), s1], dim=1))

        return self.tail(x)


# Test model
model = UNet3D(base_channels=16).to(device)
dummy = torch.zeros(1, 1, 64, 64, 64).to(device)
out = model(dummy)
print(f"Output shape: {out.shape}")

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

Output shape: torch.Size([1, 1, 64, 64, 64])
Total parameters: 5,644,913


In [6]:
class DiceBCELoss(nn.Module):
    def __init__(self, smooth=1e-5, bce_weight=50.0):
        super().__init__()
        self.smooth = smooth
        self.bce = nn.BCEWithLogitsLoss(
            pos_weight=torch.tensor([bce_weight]).to(device)
        )

    def forward(self, logits, targets):
        bce_loss = self.bce(logits, targets)

        probs = torch.sigmoid(logits).view(-1)
        targets_flat = targets.view(-1)
        intersection = (probs * targets_flat).sum()
        dice_loss = 1 - (2. * intersection + self.smooth) / (
            probs.sum() + targets_flat.sum() + self.smooth)

        return bce_loss + dice_loss

In [7]:
import shutil

local_dir = '/content/train_data'
os.makedirs(local_dir, exist_ok=True)

print("Copying data from Drive to local disk (one-time, ~2 min)...")
for subj in tqdm(all_subjects):
    src = f'{data_dir}/{subj}'
    dst = f'{local_dir}/{subj}'
    if not os.path.exists(dst):
        shutil.copytree(src, dst)

print("Done! Data is now on local disk.")

Copying data from Drive to local disk (one-time, ~2 min)...


100%|██████████| 200/200 [09:45<00:00,  2.93s/it]

Done! Data is now on local disk.


In [8]:
cache_dir = '/content/train_cache'
os.makedirs(cache_dir, exist_ok=True)

print("Converting nii.gz to numpy cache...")
for subj in tqdm(all_subjects):
    img_out = f'{cache_dir}/{subj}_image.npy'
    mask_out = f'{cache_dir}/{subj}_catheter.npy'

    if not os.path.exists(img_out):
        img = nib.load(f'/content/train_data/{subj}/{subj}_image.nii.gz').get_fdata().astype(np.float32)
        img = (img - img.min()) / (img.max() - img.min() + 1e-8)
        np.save(img_out, img)

    if not os.path.exists(mask_out):
        mask = nib.load(f'/content/train_data/{subj}/{subj}_catheter.nii.gz').get_fdata().astype(np.float32)
        mask = (mask == 1).astype(np.float32)
        np.save(mask_out, mask)

print("Done!")

Converting nii.gz to numpy cache...


100%|██████████| 200/200 [02:32<00:00,  1.31it/s]

Done!


In [9]:
print("Building train dataset...")
train_dataset = CatheterPatchDataset(
    '/content/train_cache', train_subjects,
    patch_size=64, patches_per_subject=16, augment=True
)

print("\nBuilding val dataset...")
val_dataset = CatheterPatchDataset(
    '/content/train_cache', val_subjects,
    patch_size=64, patches_per_subject=8, augment=False
)

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True,
                          num_workers=0, pin_memory=False)
val_loader = DataLoader(val_dataset, batch_size=2, shuffle=False,
                        num_workers=0, pin_memory=False)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

Building train dataset...
Scanning catheter coordinates...


100%|██████████| 160/160 [00:21<00:00,  7.50it/s]



Building val dataset...
Scanning catheter coordinates...


100%|██████████| 40/40 [00:06<00:00,  6.63it/s]

Train batches: 1280
Val batches: 160


Train on dataset (160 train/40 validation)

In [10]:
model = UNet3D(base_channels=16).to(device)
criterion = DiceBCELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

save_path = '/content/drive/MyDrive/MIA_Project_2/models/unet3d_catheter.pth'
checkpoint_dir = '/content/drive/MyDrive/MIA_Project_2/models/checkpoints'
os.makedirs(os.path.dirname(save_path), exist_ok=True)
os.makedirs(checkpoint_dir, exist_ok=True)

best_val_loss = float('inf')
epochs = 30

for epoch in range(epochs):
    # --- Training ---
    model.train()
    train_loss = 0.0
    for imgs, masks in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]"):
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        loss = criterion(model(imgs), masks)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)

    # --- Validation ---
    model.eval()
    val_loss = 0.0
    val_dice = 0.0
    with torch.no_grad():
        for imgs, masks in tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]"):
            imgs, masks = imgs.to(device), masks.to(device)
            logits = model(imgs)
            val_loss += criterion(logits, masks).item()

            preds = (torch.sigmoid(logits) > 0.5).float()
            intersection = (preds * masks).sum()
            dice = (2. * intersection + 1e-5) / (preds.sum() + masks.sum() + 1e-5)
            val_dice += dice.item()

    val_loss /= len(val_loader)
    val_dice /= len(val_loader)
    scheduler.step(val_loss)

    print(f"Epoch {epoch+1}: train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | val_dice={val_dice:.4f}")

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), save_path)
        print(f"  --> Saved best model")

    # Save checkpoint every 5 epochs
    if (epoch + 1) % 5 == 0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_val_loss': best_val_loss,
            'train_loss': train_loss,
            'val_loss': val_loss,
            'val_dice': val_dice,
        }, f'{checkpoint_dir}/checkpoint_epoch{epoch+1}.pth')
        print(f"  --> Saved checkpoint at epoch {epoch+1}")

Epoch 1/30 [Train]:  10%|▉         | 123/1280 [00:53<08:18,  2.32it/s]


KeyboardInterrupt: 

Continue training saved checkpoint model (run training setup first)



In [1]:
# Reload everything fresh
from google.colab import drive
drive.mount('/content/drive')

import os, json, numpy as np, nibabel as nib, torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

data_dir = '/content/drive/MyDrive/MIA_Project_2/Train'
all_subjects = sorted(os.listdir(data_dir))

# Rebuild train/val split (same seed = same split)
np.random.seed(42)
left_subjs, right_subjs = [], []
for subj in all_subjects:
    with open(f'{data_dir}/{subj}/{subj}.json') as f:
        meta = json.load(f)
    if meta['hemisphere'] == 'left':
        left_subjs.append(subj)
    else:
        right_subjs.append(subj)

np.random.shuffle(left_subjs)
np.random.shuffle(right_subjs)
n_left_train = int(0.8 * len(left_subjs))
n_right_train = int(0.8 * len(right_subjs))
train_subjects = left_subjs[:n_left_train] + right_subjs[:n_right_train]
val_subjects = left_subjs[n_left_train:] + right_subjs[n_right_train:]
print(f"Train: {len(train_subjects)}, Val: {len(val_subjects)}")

Mounted at /content/drive
Using device: cuda
Train: 160, Val: 40


In [11]:
checkpoint_path = '/content/drive/MyDrive/MIA_Project_2/models/checkpoints/checkpoint_epoch25.pth'
checkpoint = torch.load(checkpoint_path, map_location=device)

model = UNet3D(base_channels=16).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

start_epoch = checkpoint['epoch'] + 1
best_val_loss = checkpoint['best_val_loss']

print(f"Resumed from epoch {start_epoch}, best val loss so far: {best_val_loss:.4f}")

Resumed from epoch 25, best val loss so far: 0.2947


In [12]:
save_path = '/content/drive/MyDrive/MIA_Project_2/models/unet3d_catheter.pth'
checkpoint_dir = '/content/drive/MyDrive/MIA_Project_2/models/checkpoints'
epochs = 30

for epoch in range(start_epoch, epochs):
    model.train()
    train_loss = 0.0
    for imgs, masks in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]"):
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        loss = criterion(model(imgs), masks)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)

    model.eval()
    val_loss = 0.0
    val_dice = 0.0
    with torch.no_grad():
        for imgs, masks in tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]"):
            imgs, masks = imgs.to(device), masks.to(device)
            logits = model(imgs)
            val_loss += criterion(logits, masks).item()
            preds = (torch.sigmoid(logits) > 0.5).float()
            intersection = (preds * masks).sum()
            dice = (2. * intersection + 1e-5) / (preds.sum() + masks.sum() + 1e-5)
            val_dice += dice.item()

    val_loss /= len(val_loader)
    val_dice /= len(val_loader)
    scheduler.step(val_loss)

    print(f"Epoch {epoch+1}: train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | val_dice={val_dice:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), save_path)
        print(f"  --> Saved best model")

    if (epoch + 1) % 5 == 0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_val_loss': best_val_loss,
            'train_loss': train_loss,
            'val_loss': val_loss,
            'val_dice': val_dice,
        }, f'{checkpoint_dir}/checkpoint_epoch{epoch+1}.pth')
        print(f"  --> Saved checkpoint at epoch {epoch+1}")

Epoch 26/30 [Val]: 100%|██████████| 160/160 [00:24<00:00,  6.63it/s]


Epoch 26: train_loss=0.3195 | val_loss=0.3756 | val_dice=0.8550


Epoch 27/30 [Val]: 100%|██████████| 160/160 [00:24<00:00,  6.59it/s]


Epoch 27: train_loss=0.3260 | val_loss=0.3429 | val_dice=0.8255


Epoch 28/30 [Val]: 100%|██████████| 160/160 [00:23<00:00,  6.67it/s]


Epoch 28: train_loss=0.3119 | val_loss=0.3091 | val_dice=0.8426


Epoch 29/30 [Val]: 100%|██████████| 160/160 [00:24<00:00,  6.55it/s]


Epoch 29: train_loss=0.3043 | val_loss=0.3052 | val_dice=0.8403


Epoch 30/30 [Val]: 100%|██████████| 160/160 [00:24<00:00,  6.64it/s]


Epoch 30: train_loss=0.3226 | val_loss=0.3066 | val_dice=0.8784
  --> Saved checkpoint at epoch 30


Evaluate model

In [13]:
def sliding_window_inference(model, volume, patch_size=64, stride=32, device='cuda', threshold=0.5):
    """Run sliding window inference on a full 3D volume."""
    model.eval()
    D, H, W = volume.shape
    p = patch_size

    pred_sum = np.zeros((D, H, W), dtype=np.float32)
    count_map = np.zeros((D, H, W), dtype=np.float32)

    # Normalize volume
    vol_norm = (volume - volume.min()) / (volume.max() - volume.min() + 1e-8)
    vol_tensor = torch.tensor(vol_norm).unsqueeze(0).unsqueeze(0).float()  # (1,1,D,H,W)

    with torch.no_grad():
        for z in range(0, D - p + 1, stride):
            for y in range(0, H - p + 1, stride):
                for x in range(0, W - p + 1, stride):
                    patch = vol_tensor[:, :, z:z+p, y:y+p, x:x+p].to(device)
                    logit = model(patch)
                    prob = torch.sigmoid(logit).squeeze().cpu().numpy()
                    pred_sum[z:z+p, y:y+p, x:x+p] += prob
                    count_map[z:z+p, y:y+p, x:x+p] += 1

    # Average overlapping predictions
    count_map = np.maximum(count_map, 1)
    avg_pred = pred_sum / count_map
    return (avg_pred > threshold).astype(np.uint8)

In [14]:
# Load best model
model = UNet3D(base_channels=16).to(device)
model.load_state_dict(torch.load(
    '/content/drive/MyDrive/MIA_Project_2/models/unet3d_catheter.pth',
    map_location=device
))
model.eval()

dice_scores = []

for subj in tqdm(val_subjects):
    # Load volume and ground truth
    img = nib.load(f'/content/train_data/{subj}/{subj}_image.nii.gz').get_fdata().astype(np.float32)
    gt = nib.load(f'/content/train_data/{subj}/{subj}_catheter.nii.gz').get_fdata().astype(np.float32)
    gt = (gt == 1).astype(np.uint8)  # Task B: only value 1

    # Run inference
    pred = sliding_window_inference(model, img, patch_size=64, stride=32, device=device)

    # Compute dice
    intersection = (pred * gt).sum()
    dice = (2. * intersection + 1e-5) / (pred.sum() + gt.sum() + 1e-5)
    dice_scores.append(dice)

print(f"\nFull-volume validation results ({len(val_subjects)} subjects):")
print(f"  Mean Dice: {np.mean(dice_scores):.4f}")
print(f"  Std Dice:  {np.std(dice_scores):.4f}")
print(f"  Min Dice:  {np.min(dice_scores):.4f}")
print(f"  Max Dice:  {np.max(dice_scores):.4f}")

100%|██████████| 40/40 [02:35<00:00,  3.88s/it]


Full-volume validation results (40 subjects):
  Mean Dice: 0.8220
  Std Dice:  0.0511
  Min Dice:  0.7160
  Max Dice:  0.9063


In [15]:
# Check if worst cases correlate with hemisphere
results = []
for subj, dice in zip(val_subjects, dice_scores):
    with open(f'/content/train_data/{subj}/{subj}.json') as f:
        meta = json.load(f)
    results.append({'subject': subj, 'dice': dice, 'hemisphere': meta['hemisphere']})

results.sort(key=lambda x: x['dice'])

print("5 worst subjects:")
for r in results[:5]:
    print(f"  {r['subject']}: dice={r['dice']:.4f}, hemisphere={r['hemisphere']}")

print("\n5 best subjects:")
for r in results[-5:]:
    print(f"  {r['subject']}: dice={r['dice']:.4f}, hemisphere={r['hemisphere']}")

left_dice = [r['dice'] for r in results if r['hemisphere'] == 'left']
right_dice = [r['dice'] for r in results if r['hemisphere'] == 'right']
print(f"\nLeft hemisphere mean dice:  {np.mean(left_dice):.4f} (n={len(left_dice)})")
print(f"Right hemisphere mean dice: {np.mean(right_dice):.4f} (n={len(right_dice)})")

5 worst subjects:
  subj050: dice=0.7160, hemisphere=right
  subj092: dice=0.7335, hemisphere=right
  subj176: dice=0.7409, hemisphere=right
  subj091: dice=0.7427, hemisphere=right
  subj052: dice=0.7486, hemisphere=right

5 best subjects:
  subj181: dice=0.8794, hemisphere=right
  subj111: dice=0.8832, hemisphere=right
  subj059: dice=0.8961, hemisphere=right
  subj133: dice=0.9001, hemisphere=right
  subj021: dice=0.9063, hemisphere=right

Left hemisphere mean dice:  0.8388 (n=2)
Right hemisphere mean dice: 0.8211 (n=38)


In [16]:
test_dir = '/content/drive/MyDrive/MIA_Project_2/Test'  # adjust if needed
test_subjects = sorted(os.listdir(test_dir))
print(f"Found {len(test_subjects)} test subjects")
print(test_subjects[:5])

Found 30 test subjects
['subj201_image.nii.gz', 'subj202_image.nii.gz', 'subj203_image.nii.gz', 'subj204_image.nii.gz', 'subj205_image.nii.gz']


In [17]:
# Copy test data to local disk for speed
test_local_dir = '/content/test_data'
os.makedirs(test_local_dir, exist_ok=True)

print("Copying test data to local disk...")
for f in tqdm(test_subjects):
    src = f'{test_dir}/{f}'
    dst = f'{test_local_dir}/{f}'
    if not os.path.exists(dst):
        shutil.copy(src, dst)

print("Done!")

# Get unique subject IDs
test_ids = sorted(set([f.replace('_image.nii.gz', '') for f in test_subjects]))
print(f"\nTest subject IDs: {test_ids[:5]}")
print(f"Total: {len(test_ids)}")

Copying test data to local disk...


100%|██████████| 30/30 [00:41<00:00,  1.37s/it]

Done!

Test subject IDs: ['subj201', 'subj202', 'subj203', 'subj204', 'subj205']
Total: 30


In [18]:
output_dir = '/content/drive/MyDrive/MIA_Project_2/predictions'
os.makedirs(output_dir, exist_ok=True)

print("Running inference on test subjects...")
for subj in tqdm(test_ids):
    # Load image
    img_nib = nib.load(f'{test_local_dir}/{subj}_image.nii.gz')
    img = img_nib.get_fdata().astype(np.float32)

    # Run sliding window inference
    pred = sliding_window_inference(model, img, patch_size=64, stride=32, device=device)

    # Save prediction as nii.gz with same header/affine as input
    pred_nib = nib.Nifti1Image(pred.astype(np.uint8), affine=img_nib.affine, header=img_nib.header)
    nib.save(pred_nib, f'{output_dir}/{subj}_taskB.nii.gz')

print(f"\nDone! Saved {len(test_ids)} predictions to {output_dir}")

# Quick sanity check on first prediction
sample = nib.load(f'{output_dir}/subj201_taskB.nii.gz').get_fdata()
print(f"\nSanity check subj201:")
print(f"  Shape: {sample.shape}")
print(f"  Unique values: {np.unique(sample)}")
print(f"  Predicted catheter voxels: {int(sample.sum())}")

Running inference on test subjects...


100%|██████████| 30/30 [01:56<00:00,  3.90s/it]


Done! Saved 30 predictions to /content/drive/MyDrive/MIA_Project_2/predictions

Sanity check subj201:
  Shape: (192, 224, 192)
  Unique values: [0. 1.]
  Predicted catheter voxels: 1428


Train with all 200 subjects (run training setup first)

In [19]:
# Use all 200 subjects for final training
all_train_subjects = all_subjects  # all 200

print("Building full dataset...")
full_dataset = CatheterPatchDataset(
    '/content/train_cache', all_train_subjects,
    patch_size=64, patches_per_subject=16, augment=True
)

full_loader = DataLoader(full_dataset, batch_size=2, shuffle=True,
                         num_workers=0, pin_memory=False)

print(f"Total batches per epoch: {len(full_loader)}")

Building full dataset...
Scanning catheter coordinates...


100%|██████████| 200/200 [00:23<00:00,  8.59it/s]

Total batches per epoch: 1600


In [ ]:
model = UNet3D(base_channels=16).to(device)
criterion = DiceBCELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

save_path = '/content/drive/MyDrive/MIA_Project_2/models/unet3d_catheter_full.pth'
checkpoint_dir = '/content/drive/MyDrive/MIA_Project_2/models/checkpoints_full'
os.makedirs(os.path.dirname(save_path), exist_ok=True)
os.makedirs(checkpoint_dir, exist_ok=True)

epochs = 30

for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    for imgs, masks in tqdm(full_loader, desc=f"Epoch {epoch+1}/{epochs}"):
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        loss = criterion(model(imgs), masks)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(full_loader)

    print(f"Epoch {epoch+1}: train_loss={train_loss:.4f}")

    if (epoch + 1) % 5 == 0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': train_loss,
        }, f'{checkpoint_dir}/checkpoint_epoch{epoch+1}.pth')
        print(f"  --> Saved checkpoint at epoch {epoch+1}")

torch.save(model.state_dict(), save_path)
print("Training complete!")

Epoch 1/30: 100%|██████████| 1600/1600 [12:13<00:00,  2.18it/s]


Epoch 1: train_loss=1.4232


Epoch 2/30:  85%|████████▍ | 1358/1600 [09:58<01:51,  2.17it/s]